# 01 — Exploración y preprocesamiento de datos

Este notebook verifica la Fase 1 completa:
1. Descarga de accidentes de tránsito en Manhattan (NYC Open Data)
2. Limpieza y proyección a UTM Zone 18N
3. Descarga de polígonos de zonas prohibidas (OSM)
4. Visualización preliminar en mapa

In [2]:
import sys
sys.path.insert(0, '..')  # para importar src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

from src.data_loader import descargar_accidentes_manhattan, obtener_bboxes_osm
from src.preprocessing import limpiar_y_proyectar, guardar_procesado, obtener_puntos, muestra_estratificada

ImportError: cannot import name 'obtener_bboxes_osm' from 'src.data_loader' (c:\Users\james\OneDrive\Documents\Universidad\Semestre_8\GeometriaComputacional\ProyectoFinal\notebooks\..\src\data_loader.py)

## 1. Descarga de accidentes

In [ ]:
df_raw = descargar_accidentes_manhattan(anios=[2024], limite_total=10_000)
print(f"Filas descargadas: {len(df_raw):,}")
df_raw.head(3)

## 2. Limpieza y proyección UTM

In [ ]:
df = limpiar_y_proyectar(df_raw)
guardar_procesado(df)

print(f"\nRango latitud:  {df['latitude'].min():.4f} — {df['latitude'].max():.4f}")
print(f"Rango longitud: {df['longitude'].min():.4f} — {df['longitude'].max():.4f}")
print(f"Rango x_utm:    {df['x_utm'].min():.1f} — {df['x_utm'].max():.1f} m")
print(f"Rango y_utm:    {df['y_utm'].min():.1f} — {df['y_utm'].max():.1f} m")
df.head(3)

## 3. Verificación de muestras para Experimento 2

In [ ]:
for n in [100, 1_000, 5_000, 10_000]:
    pts = muestra_estratificada(df, n)
    print(f"n={n:>6,}  →  shape={pts.shape}")

## 4. Visualización en espacio UTM

In [ ]:
pts = obtener_puntos(df)

fig, ax = plt.subplots(figsize=(6, 10))
ax.scatter(pts[:, 0], pts[:, 1], s=1, alpha=0.3, color='steelblue')
ax.set_xlabel('x UTM (m)')
ax.set_ylabel('y UTM (m)')
ax.set_title(f'Accidentes en Manhattan 2024\n(n = {len(pts):,} puntos)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../results/figures/01_puntos_utm.png', dpi=150)
plt.show()

## 5. Descarga de zonas prohibidas (OSM)

In [ ]:
zonas = obtener_bboxes_osm()

for nombre, geom in zonas.items():
    print(f"{nombre:25s}  tipo={geom.geom_type}  área={geom.area/1e6:.2f} km²")

## 6. Mapa interactivo (folium)

In [ ]:
mapa = folium.Map(location=[40.775, -73.97], zoom_start=12, tiles='CartoDB positron')

# Muestra de 500 puntos para no saturar el mapa
muestra_vis = df.sample(500, random_state=42)
for _, fila in muestra_vis.iterrows():
    folium.CircleMarker(
        location=[fila['latitude'], fila['longitude']],
        radius=2,
        color='steelblue',
        fill=True,
        fill_opacity=0.5,
    ).add_to(mapa)

# Zonas prohibidas
colores = {'central_park': 'green', 'hudson_river': 'blue', 'east_river': 'blue'}
for nombre, geom in zonas.items():
    folium.GeoJson(
        geom.__geo_interface__,
        style_function=lambda _, c=colores.get(nombre, 'red'): {
            'fillColor': c, 'color': c, 'fillOpacity': 0.25
        },
        tooltip=nombre,
    ).add_to(mapa)

mapa